# Poker Parser | Prior to Database Design/Schema

In [2]:
from pathlib import Path
import pandas as pd
import tomllib


# Standing Jupyter Notebook in same directory as hand histories
root_folder = Path(r"C:\data\poker_hand_histories\handhq")

# Using our path, we use rglob method in our pathlib/path library to check count of phhs files.
# Then convert to list and upload to files variable
files = list(root_folder.rglob("*.phhs"))

print(f"Total files: {len(files)}")
print(files[0])

Total files: 21782
C:\data\poker_hand_histories\handhq\PTY-2009-07-01_2009-07-23_600NLH_OBFU\6\pty NLH handhq_1-OBFUSCATED.phhs


As seen above, we're using Python's standard library, rglob method which will go through every subfolder within our path and match files to our `*.phhs` pattern into a list. We assign that to our `files` variable then print the length and an example at index 0 to verify what we're working with. **21k+ files and an obfuscated file with templated intel at index Zero:** *[Site, date, stakes, game, obfuscated followed by a folder with the big blind amount]*

If you review our files (may print them for this Jupyter Notebook as well to showcase), and with some research in parsing PHHS files through Anthropic's Claude AI, I came across the TOML library. According to these sources, the PHH format's authors chose TOML as their container because it natively handles exactly what a poker hand needs: strings, numbers, booleans, arrays, and date/times as first-class types." The module is Python's built in TOML parser and its load method and returns a dictionary which we'll iterate through its keys.

We print the keys of our dictionary in a list and new line for visibility.

Then we iterate through our key/value pairs using our items() method for dictionaries and print each pair.

**Other Python Modules Considered:** There's also a pokerkit library which was built for PHH parsing which I may explore in future analysis

### Exploration only -- single-file preview
This cell just previews the structure of one file for the report narrative. It does not touch the database and is not part of the load pipeline.

In [3]:
# tomlib.load() method requires a binary file object so we're using "rb" for reading in binary
with open(files[0], "rb") as f:
    data = tomllib.load(f)

print(f"Hands in this file: {len(data)}")

first_key = next(iter(data))
hand = data[first_key] # Plain Dict Lookup
print(f"Keys: {list(hand.keys())}\n")

for k, v in hand.items():
    print(f"{k}: {v}")

Hands in this file: 998
Keys: ['variant', 'ante_trimming_status', 'antes', 'blinds_or_straddles', 'min_bet', 'starting_stacks', 'actions', 'venue', 'time', 'day', 'month', 'year', 'hand', 'seats', 'table', 'players', 'winnings', 'currency', 'currency_symbol', 'time_zone_abbreviation']

variant: NT
ante_trimming_status: False
antes: [0, 0, 0, 0, 0, 0]
blinds_or_straddles: [3, 6, 0, 0, 0, 0]
min_bet: 6
starting_stacks: [534.3, 820.1, 600, 901.48, 1301.47, 1180.5]
actions: ['d dh p1 ????', 'd dh p2 ????', 'd dh p3 ????', 'd dh p4 ????', 'd dh p5 ????', 'd dh p6 ????', 'p3 f', 'p4 f', 'p5 f', 'p6 f', 'p1 f']
venue: PartyPoker
time: 00:16:19
day: 1
month: 7
year: 2009
hand: 16405638818
seats: [4, 5, 6, 1, 2, 3]
table: Deep Stack #1417841
players: ['8C0Sx6GBydIDtuOYIwt1yA', 'zHBj5rxMtUBNSg/xGVPGJg', '/D8x6PTRypuOWrlNcuDgmg', 'NycOnR2YMNau8qul0NdUPw', 'jkR+Q4wodnyYqk5WmaypCg', 'mFhyxLamJxxZDuyoI7JK9g']
winnings: [0, 9, 0, 0, 0, 0]
currency: USD
currency_symbol: $
time_zone_abbreviation: CEST


Above

(Poker Analysis Notes) Some assumptions: parsing the actions field, and as shown in the title of each file, OBFU and the '???' in dealer actions, the hole cards are hidden so we won't be able to make determinations of hand strength unless we know the hand at showdown. Instead, for simplicity and a solid baseline we'll keep our analysis action-based (bet sizing, positions, aggression), not hand-strength based.

## Parser functions
Core parsing logic -- hand-level, action-level, and file-path metadata.

In [4]:
# Hand Level Parser
from datetime import datetime, date
import math

def clean(x):
    """inf/nan; to return none instead so we can move forward."""
    if isinstance(x, float) and not math.isfinite(x):
        return None
    return x

def parse_hand(hand_key, hand, file_meta):
    """Split one PHH hand dict into hands-row + handplayers-rows."""
    time_stamp = datetime.combine(
        date(hand["year"], hand["month"], hand["day"]),
        hand["time"],
    )
    # Mapping hands data to a variable to import to hands table
    hands_row = {
        "site_hand_id": hand["hand"],
        "variant": hand["variant"],
        "site": hand["venue"],
        "tble_name": hand.get("table"),
        "played_at": time_stamp,
        "time_zone": hand.get("time_zone_abbreviation"),   # .get, not logged across sites
        "min_bet": hand["min_bet"],
        "n_players": len(hand["players"]),
        **file_meta,      # site, stake, blind_level from the folder path
    }

    hp_rows = [] # Same thing here for the players/hand table using iterating logic to ensure positions
    for i, pid in enumerate(hand["players"]):
        antes = hand.get("antes", [0] * len(hand["players"]))
        winnings = hand.get("winnings", [None] * len(hand["players"])) # using get methods due to errors retreving data
        hp_rows.append({
            "site_hand_id": hand["hand"],
            "player_token": pid,
            "position_index": i + 1,
            "seat_no": hand["seats"][i],
            "starting_stack": clean(hand["starting_stacks"][i]),
            "blind_posted": clean(hand["blinds_or_straddles"][i]),
            "ante": clean(antes[i]),
            "winnings": clean(winnings[i]),
        })
    return hands_row, hp_rows

In [5]:
STREETS = ['preflop', 'flop', 'turn', 'river']

def parse_actions(actions, bb_position=2): # Bb starts at 2
    """Parse PHH action list into rows with street tracking and check/call resolution."""
    rows = []
    street_idx = 0
    raised = False # has anyone cbr'd on the current street? Default is False

    for order, raw in enumerate(actions):
        parts = raw.split() # Split out actions
        actor = parts[0]

        if actor == 'd': # Dealer parsing
            if parts[1] == 'db': # if the next part is deals board 'db' push the street up one (to the flop, street, river)
                street_idx += 1
                raised = False # new street resets betting
                rows.append({
                    'action_order': order,
                    'actor': 'dealer',
                    'street': STREETS[street_idx],
                    'action_type': 'deal_board', # dealer dealing
                    'cards': parts[2],
                    'amount': None,
                })
            elif parts[1] == 'dh': # deals hole cards to players
                rows.append({
                    'action_order': order,
                    'actor': 'dealer',
                    'street': STREETS[street_idx],
                    'action_type': 'deal_hole',
                    'cards': parts[3] if len(parts) > 3 else parts[2],
                    'amount': None,
                })
        else: # Else then we're looking at player action types
            action_type = parts[1]

            if action_type == 'cc':
                if street_idx == 0:
                    # preflop: blinds are live bets -> cc is a call,
                    # EXCEPT the big blind checking their option (no raise occurred)
                    is_bb_option = (actor == f'p{bb_position}' and not raised)
                    action_type = 'check' if is_bb_option else 'call'
                else:
                    # postflop: cc is a check until someone bets this street
                    action_type = 'call' if raised else 'check'
            elif action_type == 'cbr':
                raised = True

            amount = None # Amount default zero/nullable
            if len(parts) > 2: # 3 parts Ex. 'p1 cbr 300'
                try:
                    amount = float(parts[2])
                except ValueError:
                    pass

            rows.append({
                'action_order': order,
                'actor': actor,
                'street': STREETS[street_idx],
                'action_type': action_type,
                'amount': amount,
                'cards': None,
            })

    return rows

In [6]:
def parse_path_metadata(f: Path):
    '''Parsing the info from out file path metadata'''
    session_dir = f.relative_to(root_folder).parts[0] # 'ABS-2009-07-01_2009-07-23_50NLH_OBFU'
    blind_dir   = f.relative_to(root_folder).parts[1] # '0.5'

    parts = session_dir.split("_") # ['ABS-2009-07-01', '2009-07-23', '50NLH', 'OBFU']
    site, date_start = parts[0].split("-", 1) # 'ABS', '2009-07-01'
    date_end, stake  = parts[1], parts[2] # '2009-07-23', '50NLH'

    return {
        "site": site,
        "date_start": date_start,
        "date_end": date_end,
        "stake": stake,
        "blind_level": float(blind_dir),
        "filename": f.name,
    }

print(parse_path_metadata(files[0]))

{'site': 'PTY', 'date_start': '2009-07-01', 'date_end': '2009-07-23', 'stake': '600NLH', 'blind_level': 6.0, 'filename': 'pty NLH handhq_1-OBFUSCATED.phhs'}


## File sampling
Build the candidate file lists per stake, then sample and choose how many files to actually load.

In [7]:
from collections import defaultdict

# Building our stake from/session data from the relative paths/parts of the folder names.
stake_files = defaultdict(list)
for f in root_folder.rglob("*.phhs"):
    session = f.relative_to(root_folder).parts[0] # 'ABS-2009-..._50NLH_OBFU'
    stake = session.split("_")[2] # '50NLH'
    if stake in ("50NLH", "200NLH", "1000NLH"): # We are going to assess these 3 different games to show the biggest differences
        stake_files[stake].append(f)

for stake, fl in stake_files.items():
    print(f"{stake}: {len(fl)} files ~= {len(fl) * 998:,} hands") # 998 hands as baseline

1000NLH: 2219 files ~= 2,214,562 hands
50NLH: 2147 files ~= 2,142,706 hands
200NLH: 3328 files ~= 3,321,344 hands


In [8]:
import random
random.seed(42) # reproducibility

SAMPLE_PER_STAKE = 500 # up to 500 files per stake made available to sample from
sampled = {
    stake: random.sample(fl, min(SAMPLE_PER_STAKE, len(fl)))
    for stake, fl in stake_files.items()
}
for stake, fl in sampled.items():
    print(f"{stake}: {len(fl)} files ~= {len(fl)*998:,} hands")

1000NLH: 500 files ~= 499,000 hands
50NLH: 500 files ~= 499,000 hands
200NLH: 500 files ~= 499,000 hands


## Database connection & state sync
Always run this cell before the load loop below, and always re-run it after a kernel restart. It pulls every piece of resumable state from MySQL itself (known players, next hand ID, hands already loaded, files already loaded), so the loop stays idempotent no matter how many times or in what order you run it.

In [9]:
import mysql.connector
conn = mysql.connector.connect(
    host="localhost", user="root", password="root", database="poker_database"
)
cursor = conn.cursor()
cursor.execute("SHOW TABLES")
print(cursor.fetchall())

# One-time table to track which files have already been loaded,
# so reruns can't silently double-insert the same hands.
cursor.execute("""
    CREATE TABLE IF NOT EXISTS loaded_files (
        filename VARCHAR(255) PRIMARY KEY,
        loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
conn.commit()

# Sync player_ids from what's already in the DB (don't start from empty)
cursor.execute("SELECT player_id, player_token FROM players")
player_ids = {token: pid for pid, token in cursor.fetchall()}

# Sync next_hand_id from the current max in the DB (don't start from 1)
cursor.execute("SELECT COALESCE(MAX(hand_id), 0) FROM hands")
next_hand_id = cursor.fetchone()[0] + 1

# Sync every site_hand_id already in the DB -- the real dedup guard,
# works regardless of which files loaded them or when
cursor.execute("SELECT site_hand_id FROM hands")
existing_hand_ids = {row[0] for row in cursor.fetchall()}

# Load the set of already-processed filenames
cursor.execute("SELECT filename FROM loaded_files")
already_loaded = {row[0] for row in cursor.fetchall()}

print(f"Resuming: {len(player_ids)} known players, "
      f"next_hand_id = {next_hand_id}, "
      f"{len(existing_hand_ids)} hands already loaded, "
      f"{len(already_loaded)} files already loaded")

[('actions',), ('hand_players',), ('hands',), ('loaded_files',), ('players',)]
Resuming: 21836 known players, next_hand_id = 190083, 190082 hands already loaded, 191 files already loaded


## Main load loop
Choose how many files per stake to load, then run. Safe to re-run any time: already-loaded files are skipped by name, and any individual duplicate hand is skipped by `site_hand_id` even within a fresh file.

In [10]:
files_to_load = []
skipped_hands = []

for stake, fl in sampled.items():
    files_to_load.extend(fl[:75])   # adjust this number to scale up/down

for path in files_to_load:
    if path.name in already_loaded:
        print(f"skip (already loaded): {path.name}")
        continue

    file_meta = parse_path_metadata(path)
    with open(path, "rb") as f:
        data = tomllib.load(f)

    new_players, hands_t, hp_t, act_t = [], [], [], []

    for key, hand in data.items():
        try:
            site_hand_id = hand["hand"]
        except KeyError:
            skipped_hands.append((file_meta["filename"], key, "missing 'hand' key"))
            continue
        if site_hand_id in existing_hand_ids:
            continue

        h_row, hp_rows = parse_hand(key, hand, file_meta)
        bb_pos = hand["blinds_or_straddles"].index(max(hand["blinds_or_straddles"])) + 1
        a_rows = parse_actions(hand["actions"], bb_position=bb_pos)

        hid = next_hand_id
        next_hand_id += 1
        existing_hand_ids.add(site_hand_id)

        hands_t.append((
            hid, h_row["site_hand_id"], h_row["variant"], h_row["site"],
            h_row["stake"], h_row["blind_level"], h_row["tble_name"],
            h_row["played_at"], h_row["time_zone"], h_row["min_bet"],
            h_row["n_players"],
        ))

        for r in hp_rows:
            tok = r["player_token"]
            if tok not in player_ids:
                player_ids[tok] = len(player_ids) + 1
                new_players.append((player_ids[tok], tok))
            hp_t.append((
                hid, player_ids[tok], r["position_index"], r["seat_no"],
                r["starting_stack"], r["blind_posted"], r["ante"], r["winnings"],
            ))

        for r in a_rows:
            act_t.append((
                hid, r["action_order"], r["actor"], r["street"],
                r["action_type"], r["amount"], r["cards"],
            ))

    if new_players:
        cursor.executemany(
            "INSERT INTO players (player_id, player_token) VALUES (%s, %s)",
            new_players)
    cursor.executemany("""
        INSERT INTO hands (hand_id, site_hand_id, variant, site, stake,
                           blind_level, tble_name, played_at, time_zone,
                           min_bet, n_players)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""", hands_t)
    cursor.executemany("""
        INSERT INTO hand_players (hand_id, player_id, position_index, seat_no,
                                  starting_stack, blind_posted, ante, winnings)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s)""", hp_t)
    cursor.executemany("""
        INSERT INTO actions (hand_id, action_order, actor, street,
                             action_type, amount, cards)
        VALUES (%s,%s,%s,%s,%s,%s,%s)""", act_t)

    cursor.execute("INSERT INTO loaded_files (filename) VALUES (%s)", (path.name,))
    conn.commit()  # one commit per file: crash loses at most one file's worth of work
    already_loaded.add(path.name)   # keep in-memory set in sync with the DB, not just at session start
    print(f"{path.name}: {len(hands_t)} hands, {len(hp_t)} seats, {len(act_t)} actions")

skip (already loaded): ps NLH handhq_111-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_191-OBFUSCATED.phhs
skip (already loaded): ipn NLH handhq_133-OBFUSCATED.phhs
skip (already loaded): ong NLH handhq_314-OBFUSCATED.phhs
skip (already loaded): ong NLH handhq_234-OBFUSCATED.phhs
skip (already loaded): ps NLH handhq_215-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_79-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_42-OBFUSCATED.phhs
skip (already loaded): ipn NLH handhq_676-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_216-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_209-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_46-OBFUSCATED.phhs
skip (already loaded): ong NLH handhq_217-OBFUSCATED.phhs
skip (already loaded): ong NLH handhq_269-OBFUSCATED.phhs
skip (already loaded): ftp NLH handhq_369-OBFUSCATED.phhs
skip (already loaded): pty NLH handhq_197-OBFUSCATED.phhs
skip (already loaded): ong NLH handhq_144-OBFUSCATED.phhs
skip (already loade